Woah sick

In [1]:
!pip install pyspark pyarrow

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
print(os.getcwd())

/expanse/lustre/projects/uci157/tragus


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType, DoubleType, FloatType
from pyspark.sql.functions import col, count, length,countDistinct, min as spark_min, max as spark_max,avg, stddev, approx_count_distinct

spark = (SparkSession.builder.appName("MusicBrainz").config("spark.driver.memory", "2g").config("spark.executor.memory", "18g").config('spark.executor.instances', 7).getOrCreate())

I saw on Piazza somebody was asking about the Spark Jobs UI, and apparently I was right that this is something distinct from the Expanse Jobs page. Apparently it's this-

Not sure how to get it to display the active number of tasks since it's not live

In [4]:
import requests
import pandas as pd

# Get the active Spark Context and URL
sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

# Fetch the executor data from the API
response = requests.get(url)
executors = response.json()

# Format into a readable DataFrame
spark_df = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
spark_df['maxMemory_GB'] = (spark_df['maxMemory'] / (1024**3)).round(2)
spark_df

,id,totalCores,maxMemory,activeTasks,isActive,maxMemory_GB
0,driver,8,1099746508,0,True,1.02


In [5]:
MBDUMP = "musicbrainz_project/raw_data/mbdump"

In [6]:
def peek(df, n=3):
    return display(df.limit(n).toPandas())

# DEFINE SCHEMAS

In [7]:
artist_schema = StructType([
    StructField("id",               IntegerType(),   True),
    StructField("gid",              StringType(),    True),
    StructField("name",             StringType(),    True),
    StructField("sort_name",        StringType(),    True),
    StructField("begin_date_year",  IntegerType(),   True),
    StructField("begin_date_month", IntegerType(),   True),
    StructField("begin_date_day",   IntegerType(),   True),
    StructField("end_date_year",    IntegerType(),   True),
    StructField("end_date_month",   IntegerType(),   True),
    StructField("end_date_day",     IntegerType(),   True),
    StructField("type",             IntegerType(),   True),
    StructField("area",             IntegerType(),   True),
    StructField("gender",           IntegerType(),   True),
    StructField("comment",          StringType(),    True),
    StructField("edits_pending",    IntegerType(),   True),
    StructField("last_updated",     StringType(),    True),
    StructField("ended",            StringType(),    True),
    StructField("begin_area",       IntegerType(),   True),
    StructField("end_area",         IntegerType(),   True),
])
instrument_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("gid",           StringType(),  True),
    StructField("name",          StringType(),  True),
    StructField("type",          IntegerType(), True),
    StructField("edits_pending", IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
])
label_schema = StructType([
    StructField("id",                IntegerType(), True),
    StructField("gid",               StringType(),  True),
    StructField("name",              StringType(),  True),
    StructField("begin_date_year",   IntegerType(), True),
    StructField("begin_date_month",  IntegerType(), True),
    StructField("begin_date_day",    IntegerType(), True),
    StructField("end_date_year",     IntegerType(), True),
    StructField("end_date_month",    IntegerType(), True),
    StructField("end_date_day",      IntegerType(), True),
    StructField("label_code",        IntegerType(), True),
    StructField("type",              IntegerType(), True),
    StructField("area",              IntegerType(), True),
    StructField("comment",           StringType(),  True),
    StructField("edits_pending",     IntegerType(), True),
    StructField("last_updated",      StringType(),  True),
    StructField("ended",             StringType(),  True),
])
genre_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("gid",           StringType(),  True),
    StructField("name",          StringType(),  True),
    StructField("comment",       StringType(),  True),
    StructField("edits_pending", IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
])
area_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("gid",           StringType(),  True),
    StructField("name",          StringType(),  True),
    StructField("type",          IntegerType(), True),
    StructField("comment",       StringType(),  True),
    StructField("edits_pending", IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
])
tag_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("name",          StringType(),  True),
    StructField("ref_count",     IntegerType(), True),
])
gender_schema = StructType([
    StructField("id",   IntegerType(), True),
    StructField("name", StringType(),  True),
])
release_group_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("gid",           StringType(),  True),
    StructField("name",          StringType(),  True),
    StructField("artist_credit", IntegerType(), True),
    StructField("type",          IntegerType(), True),
    StructField("comment",       StringType(),  True),
    StructField("edits_pending", IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
])
l_artist_label_schema = StructType([
    StructField("id",        IntegerType(), True),
    StructField("link",      IntegerType(), True),
    StructField("entity0",   IntegerType(), True),  # artist
    StructField("entity1",   IntegerType(), True),  # label
])
l_artist_release_group_schema = StructType([
    StructField("id",        IntegerType(), True),
    StructField("link",      IntegerType(), True),
    StructField("entity0",   IntegerType(), True),  # artist
    StructField("entity1",   IntegerType(), True),  # release_group
])
label_tag_schema = StructType([
    StructField("label", IntegerType(), True),
    StructField("tag",   IntegerType(), True),
    StructField("count", IntegerType(), True),
])
l_artist_genre_schema = StructType([
    StructField("id",             IntegerType(), True),
    StructField("link",           IntegerType(), True),
    StructField("entity0",        IntegerType(), True),  # artist
    StructField("entity1",        IntegerType(), True),
])
l_artist_artist_schema = StructType([
    StructField("id",      IntegerType(), True),
    StructField("link",    IntegerType(), True),
    StructField("entity0", IntegerType(), True),
    StructField("entity1", IntegerType(), True),
])
release_group_tag_schema = StructType([
    StructField("release_group", IntegerType(), True),
    StructField("tag",           IntegerType(), True),
    StructField("count",         IntegerType(), True),
])
artist_tag_schema = StructType([
    StructField("artist", IntegerType(), True),
    StructField("tag",    IntegerType(), True),
    StructField("count",  IntegerType(), True),
])
artist_credit_schema = StructType([
    StructField("id",         IntegerType(), True),
    StructField("name",       StringType(),  True),
    StructField("artist_count", IntegerType(), True),
])
artist_credit_name_schema = StructType([
    StructField("artist_credit", IntegerType(), True),
    StructField("position",      IntegerType(), True),
    StructField("artist",        IntegerType(), True),
    StructField("name",          StringType(),  True),
    StructField("join_phrase",   StringType(),  True),
])
l_artist_instrument_schema = StructType([
    StructField("id",             IntegerType(), True),
    StructField("link",           IntegerType(), True),
    StructField("entity0",        IntegerType(), True),  # artist
    StructField("entity1",        IntegerType(), True),  # instrument
    StructField("edits_pending",  IntegerType(), True),
    StructField("last_updated",   StringType(),  True),
    StructField("link_order",     IntegerType(), True),
    StructField("entity0_credit", StringType(),  True),
    StructField("entity1_credit", StringType(),  True),
])

So the schema definitely aren't perfect - after testing out some SQL queries it becomes clear some of variables aren't assigned to the right columns. I think it will take some trial and error to suss out which variables are assigned correctly. It's a really great start though!! I've adjusted a couple and I'll spend a little more time experimenting with SQL to improve it

Something to be aware of is that a lot of these schema don't have all of the columns present in the TSV. We can still create the dataframes bc if there are extra columns without assignment in the schema, Spark just ignores them. Usually the variables we care about appear in the first half of the dataframe so ... maybe not so important but just FYI

# BUILD TABLES

In [8]:
schemas = {
    "artist": artist_schema,
    "instrument": instrument_schema,
    "label": label_schema,
    "genre": genre_schema,
    "area": area_schema,
    "tag": tag_schema,
    "gender": gender_schema,
    "release_group": release_group_schema,
    "l_artist_label": l_artist_label_schema,
    "l_artist_release_group": l_artist_release_group_schema,
    "label_tag": label_tag_schema,
    "l_artist_genre": l_artist_genre_schema,
    "l_artist_artist": l_artist_artist_schema,
    "release_group_tag": release_group_tag_schema,
    "artist_tag": artist_tag_schema,
    "artist_credit": artist_credit_schema,
    "artist_credit_name": artist_credit_name_schema,
    "l_artist_instrument": l_artist_instrument_schema,
}

dfs = {}
profile_rows = []

for table_name, schema in schemas.items():
    df = (
        spark.read
        .option("sep", "\t")
        .option("nullValue", r"\N")
        .option("header", "false")
        .option("quote", "")
        .option("escape", "")
        .schema(schema)
        .csv(f"{MBDUMP}/{table_name}")
    )

    dfs[table_name] = df

    print(f"=== PEEK: {table_name} ===")
    peek(df)

=== PEEK: artist ===


,id,gid,name,sort_name,begin_date_year,begin_date_month,begin_date_day,end_date_year,end_date_month,end_date_day,type,area,gender,comment,edits_pending,last_updated,ended,begin_area,end_area
0,2252039,fadeb38c-833f-40bc-9d8c-a6383b38b1be,Доктор Сатана,Доктор Сатана,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2021-11-23 07:08:52.479537+00,f,NaN,NaN
1,371203,49add228-eac5-4de8-836c-d75cde7369c3,Pete Moutso,"Moutso, Pete",NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,None,0,None,f,NaN,NaN
2,3087346,dfdce491-133d-4e9f-9e48-795587e181b0,UNlT,UNlT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2025-09-14 23:53:28.004798+00,f,NaN,NaN


=== PEEK: instrument ===


,id,gid,name,type,edits_pending,last_updated
0,687,c1dbb66d-2356-417a-81ad-f688fee33257,guitarrón mexicano,2,0,2015-02-15 07:32:32.570132+00
1,695,2474c241-d267-433a-a404-688b13c51d11,jouhikko,2,0,2015-02-25 19:47:04.610414+00
2,701,c0cc863c-ea65-4b8a-b365-28b81b72d846,friction idiophone,3,0,2015-02-26 10:28:49.766548+00


=== PEEK: label ===


...

,id,name,artist_count
0,4229350,"Jean-Paul Fouchécourt, Yvonne Naef, Saito Kine...",4
1,3320885,The Turns,1
2,3320887,Son.Sine,1


=== PEEK: artist_credit_name ===


,artist_credit,position,artist,name,join_phrase
0,578352,0,578352,Gustav Ruppke,None
1,273232,0,273232,Zachary,None
2,153193,0,153193,The High Level Ranters,None


=== PEEK: l_artist_instrument ===


,id,link,entity0,entity1,edits_pending,last_updated,link_order,entity0_credit,entity1_credit
0,1,311992,1218252,121,0,2016-08-01 11:22:19.207793+00,0,None,None
1,2,312066,1395197,19,0,2016-08-01 14:14:07.499778+00,0,None,None
2,24,393478,1261488,808,0,2017-05-31 08:31:04.355584+00,0,None,None


# FOREIGN KEY MAP (for navigating the database)

In [9]:
foreign_key_map = {
    "artist": {
        "type":       ("artist_type", "id"),
        "area":       ("area", "id"),
        "gender":     ("gender", "id"),
        "begin_area": ("area", "id"),
        "end_area":   ("area", "id"),
    },
    "instrument": {
        "type": ("instrument_type", "id"),
    },
    "label": {
        "type": ("label_type", "id"),
        "area": ("area", "id"),
    },
    "release_group": {
        "type":          ("release_group_primary_type", "id"),
        "artist_credit": ("artist_credit", "id"),
    },
    "artist_credit_name": {
        "artist_credit": ("artist_credit", "id"),
        "artist":        ("artist", "id"),
    },
    "label_tag": {
        "label": ("label", "id"),
        "tag":   ("tag", "id"),
    },
    "artist_tag": {
        "artist": ("artist", "id"),
        "tag":    ("tag", "id"),
    },
    "release_group_tag": {
        "release_group": ("release_group", "id"),
        "tag":           ("tag", "id"),
    },
    "l_artist_genre": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("genre", "id"),
    },
    "l_release_group_genre": { 
        "link":    ("link", "id"),
        "entity0": ("release_group", "id"),
        "entity1": ("genre", "id"),
    },
    "l_artist_label": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("label", "id"),
    },
    "l_artist_release_group": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("release_group", "id"),
    },
    "l_artist_artist": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("artist", "id"),
    },
    "l_artist_instrument": {
        "link":    ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("instrument", "id"),
    },
}

Great work on the map! I just removed "artist_genre" and "release_group_genre" because I couldn't find those tables in the database. But I'm pretty sure it's correct

# TESTING SOME BASIC SQL QUERIES

Here are some cool queries

In [10]:
for table_name, df in dfs.items():
    df.createOrReplaceTempView(table_name)

In [11]:
def peek(df, n=20):
    return display(df.limit(n).toPandas())

***

1) Select all from artist

In [12]:
peek(spark.sql("""
    SELECT *
    FROM artist
"""))

,id,gid,name,sort_name,begin_date_year,begin_date_month,begin_date_day,end_date_year,end_date_month,end_date_day,type,area,gender,comment,edits_pending,last_updated,ended,begin_area,end_area
0,2252039,fadeb38c-833f-40bc-9d8c-a6383b38b1be,Доктор Сатана,Доктор Сатана,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2021-11-23 07:08:52.479537+00,f,NaN,NaN
1,371203,49add228-eac5-4de8-836c-d75cde7369c3,Pete Moutso,"Moutso, Pete",NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,None,0,None,f,NaN,NaN
2,3087346,dfdce491-133d-4e9f-9e48-795587e181b0,UNlT,UNlT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2025-09-14 23:53:28.004798+00,f,NaN,NaN
3,2851271,165a49a0-2b3b-4078-a3c1-905afdc07c0a,Babyglock,Babyglock,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2024-10-19 03:33:55.474151+00,f,NaN,NaN
4,145773,7b4a548e-a01a-49b7-82e7-b49efeb9732c,Aric Leavitt,"Leavitt, Aric",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,None,f,NaN,NaN
5,1076328,60aca66f-e91a-4cb5-9308-b6e293cd833e,Fonograff,Fonograff,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2014-01-10 16:25:20.992213+00,f,NaN,NaN
6,1172876,3e1bd546-d2a7-49cb-b38d-d70904a1d719,Al Street,"Street, Al",NaN,NaN,NaN,NaN,NaN,NaN,1.0,222.0,1.0,None,0,2014-11-23 14:07:19.782509+00,f,NaN,NaN
7,220155,df120895-f6c6-4a66-b9cf-73350f0beb61,Love .45,Love .45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,None,f,NaN,NaN
8,618464,c14f8d3f-ee81-416f-800f-8eff7e77a2e1,Sintellect,Sintellect,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2009-05-23 09:41:53.269195+00,f,NaN,NaN
9,285714,b68a3969-319a-462f-942b-cd35581414fc,Evie Tamala,Evie Tamala,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,None,f,NaN,NaN


***

2. Select all artists and their labels

In [13]:
peek(spark.sql("""
    SELECT a.name AS artist_name, l.name AS label
    FROM l_artist_label lal
    JOIN artist a ON lal.entity0 = a.id
    JOIN label l ON lal.entity1 = l.id
"""))

,artist_name,label
0,s'Poom,Legendaarne Records
1,Guido Elmi,Nopop
2,Marco Resmann,Upon.You
3,Karl-Jonas Winqvist,Sing a Song Fighter
4,Stuart Brown,British Film Institute
5,Earl Young,"Baker, Harris & Young Productions"
6,Alexia Coley,Jalapeno Records
7,Darren Hickey,Xpressive
8,Darren Hickey,Juice Records
9,Ras Sheehama,African Cream Music


***

3. Number of release groups per artist sorted by most prolific

In [14]:
peek(spark.sql("""
    SELECT a.name AS artist_name, COUNT(rg.id) AS release_group_count
    FROM release_group rg
    JOIN artist_credit_name acn ON rg.artist_credit = acn.artist_credit
    JOIN artist a ON acn.artist = a.id
    GROUP BY a.name
    ORDER BY release_group_count DESC
"""))

,artist_name,release_group_count
0,Various Artists,281842
1,Ludwig van Beethoven,6088
2,Johann Sebastian Bach,6011
3,Wolfgang Amadeus Mozart,5922
4,[unknown],3432
5,Johannes Brahms,2946
6,Franz Schubert,2852
7,Пётр Ильич Чайковский,2684
8,Antonio Vivaldi,2143
9,Bruce Springsteen,2127


***

4. Number of labels per country

In [15]:
peek(spark.sql("""
    SELECT a.name as area, COUNT(l.id) as label_count
    FROM label l
    JOIN area a ON l.area = a.id
    GROUP BY a.name
    ORDER BY label_count DESC
    LIMIT 20
"""))

,area,label_count
0,United States,30680
1,United Kingdom,14758
2,Japan,12059
3,Germany,9388
4,France,5974
5,Italy,4110
6,Canada,3227
7,Sweden,2933
8,Netherlands,2893
9,Spain,2345


***

5. Number of artists by gender

In [20]:
peek(spark.sql("""
    SELECT g.name AS gender, COUNT(*) AS count
    FROM artist a
    JOIN gender g ON a.gender = g.id
    GROUP BY g.name
    ORDER BY count DESC
"""))

,gender,count
0,Male,932192
1,Female,280203
2,Not applicable,2443
3,Non-binary,2237
4,Other,1787
